# 01 · Data quality and the market-time contract

Start here. This notebook reproduces the data preparation behind both dashboards, checks source integrity, and makes missing values explicit. Run all cells in order from the repository root or the `notebooks/` directory.

**Setup:** `python3.12 -m venv .venv` then `.venv/bin/python -m pip install -r requirements-dev.txt`. Select that virtual environment as the notebook kernel. No GPU or TensorFlow is required.

Market times are assumed fixed AEST (UTC+10); weather explicitly labelled UTC is converted. Data covers NSW only, 1 January 2022–24 July 2024. [Methodology](../docs/METHODOLOGY.md) · [Original notebooks](../archive/notebooks/README.md).

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'nem_analysis.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from nem_analysis import load_market, filter_market, aggregate_market, smooth_prices, price_episodes, score_forecasts, season_labels
market, metadata = load_market()
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = ['#087e72', '#df9562', '#8296b0', '#233c35']
print(f"{len(market):,} half-hour positions | {market.index.min()}–{market.index.max()} | AEST (UTC+10)")

44,911 half-hour positions | 2022-01-01 00:00:00–2024-07-24 15:00:00 | AEST (UTC+10)


## Source inventory
The duplicate solar timestamps are alternative production estimates, not extra generation. The premerged CSV is intentionally excluded.

In [2]:
display(pd.DataFrame(metadata['quality']['sources']))
display(pd.DataFrame(metadata['quality']['missing'].items(), columns=['channel', 'missing_half_hours']))

,file,rows,uniqueTimestamps,sha256
0,nsw_prices.csv,269452,269452,ca5d1cb1cc73bb102e3d9caffb62e4a3fd016fb6fe93c8...
1,nsw_demand_actual.csv,44911,44911,0221d27457f770ed05e70823672ec49ce888e2f91efe9c...
2,nsw_demand_forecast.csv,45313,45313,b0fc7471910faa8bf73c6935a080ac2cba1aea7bd8e471...
3,nsw_solar.csv,89751,44907,bef377d83a1a24d81740527accba57ef5600b97d2fe77c...
4,CCtemps.csv,22752,22752,7f21b6475cabe2ea0d95cf89adbd0ce0874b8bf39684c2...


,channel,missing_half_hours
0,price,4
1,demand,0
2,forecastDemand,0
3,solar,4
4,temperature,350
5,maxPrice,4
6,minPrice,4


## Recreate the cleaned grid from the raw files
Each half-hour price averages exactly six five-minute observations in `(t − 30 minutes, t]`. Incomplete intervals remain missing. Weather carry-forward is limited to one hour. No price clipping or backward filling occurs.

In [3]:
from scripts.build_data import prepare_data, make_features
cleaned, audit = prepare_data()
assert cleaned.index.is_unique and cleaned.index.is_monotonic_increasing
assert cleaned.index.to_series().diff().dropna().eq(pd.Timedelta('30min')).all()
assert len(cleaned) == len(market)
np.testing.assert_allclose(cleaned.price, market.price, atol=.00051, equal_nan=True)
print('Raw preparation agrees with the published dashboard artifact.')
display(cleaned.head(8))

Raw preparation agrees with the published dashboard artifact.


,price,demand,forecastDemand,solar,temperature,maxPrice,minPrice
2022-01-01 00:00:00,NaN,7329,7336,0.0,NaN,NaN,NaN
2022-01-01 00:30:00,122.755000,7145,7151,0.0,NaN,126.02,113.54
2022-01-01 01:00:00,114.468333,6891,6899,0.0,21.7,126.24,93.63
2022-01-01 01:30:00,99.680000,6578,6585,0.0,21.7,107.80,91.37
2022-01-01 02:00:00,92.440000,6306,6328,0.0,21.5,100.30,85.00
2022-01-01 02:30:00,89.235000,6163,6115,0.0,21.5,97.31,74.39
2022-01-01 03:00:00,107.058333,6057,6048,0.0,21.1,107.80,105.43
2022-01-01 03:30:00,100.756667,6001,5969,0.0,21.1,107.80,88.28


## Check source hashes and timestamp alignment

In [4]:
import hashlib
for source in audit['sources']:
    assert hashlib.sha256((ROOT / source['file']).read_bytes()).hexdigest() == source['sha256']
assert np.isnan(cleaned.iloc[0].price), 'The first partial price bin must remain missing.'
assert cleaned.loc['2022-01-01 01:00', 'temperature'] == 21.7
assert audit['solarAlternativesResolved'] == 44844
print('Source checksums, incomplete-bin handling, UTC conversion and solar alternatives checked.')

Source checksums, incomplete-bin handling, UTC conversion and solar alternatives checked.


## A direct test for future-information leakage
Changing every observed value at and after an origin must not change any feature available at that origin.

In [5]:
features = make_features(cleaned)
changed = cleaned.copy()
origin = pd.Timestamp('2023-06-01 12:00')
changed.loc[origin:, ['price','demand','solar','temperature']] = 999999
pd.testing.assert_frame_equal(features.loc[:origin], make_features(changed).loc[:origin])
print('All features at the origin depend on prior observations or known calendar values.')
display(features.loc[origin].to_frame('feature_value'))

All features at the origin depend on prior observations or known calendar values.


,feature_value
price_lag_1,2.882000e+01
price_lag_2,3.979500e+01
price_lag_48,5.726000e+01
price_lag_336,1.142200e+02
price_mean_6,5.604000e+01
price_mean_48,9.961229e+01
demand_lag_1,7.932000e+03
solar_lag_1,1.585917e+03
temperature_lag_1,1.740000e+01
hour_sin,1.224647e-16


## What remains uncertain
The source exports omit electricity timezone declarations and publication/forecast issue times. AEST is an explicit assumption, not proof of provenance. This is descriptive research and a nominal rolling backtest, not a verified operational replay. Proceed to [02 · Market exploration](02_market_exploration.ipynb).